# W6C2: One LSTM, three jobs

Run every cell from the top. **Everything already works.**

We are not building an LSTM today, we are using one. Three small models are already trained and waiting in `data/`, so every cell runs in about a second. Watch for how little separates the three.

Today you will:

1. See what `nn.LSTM` hands back, and what the two return values are for.
2. **Job 1**: generate a headline one character at a time.
3. **Job 2**: label every token in a sentence.
4. **Job 3**: decide whether a whole headline is sarcastic.
5. Then train your own sarcasm detector, in teams.

Nothing to submit. Answers are in the last cell.

In [ ]:
# Setup. Run this cell first.
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

headlines = pd.read_csv("data/headlines.csv")

print(len(headlines), "headlines")
print()
for label, name in [(1, "SARCASTIC (The Onion)"), (0, "NOT     (HuffPost)")]:
    print(name)
    for text in headlines[headlines["is_sarcastic"] == label]["headline"].head(2):
        print("   ", text)

## Part 1. What nn.LSTM gives you


An LSTM reads a sequence one step at a time and keeps a running memory. PyTorch
gives you the whole thing in one line, and it hands back **two** results.

Those two return values are the only thing you need today, because they are what
the three jobs are built from.


<img src="images/read-in-order.png" width="640">

In [ ]:
lstm = nn.LSTM(input_size=16, hidden_size=32, batch_first=True)

# A batch of 4 sequences, 10 steps long, 16 numbers per step.
sequence = torch.randn(4, 10, 16)

outputs, (last_hidden, last_cell) = lstm(sequence)

print("in                ", tuple(sequence.shape),    " (batch, time, features)")
print("outputs           ", tuple(outputs.shape),     " one vector PER STEP")
print("last_hidden       ", tuple(last_hidden.shape), " one vector for the WHOLE sequence")
print()
print("outputs[:, -1] is the same thing as last_hidden[-1]:",
      torch.allclose(outputs[:, -1], last_hidden[-1]))


That is the whole API, and it is also the whole lesson:

- use **`outputs`**, one vector per step, and you can emit something at every
  position: the next character (job 1) or a tag (job 2),
- use **`last_hidden`**, one vector for the whole sequence, and you can emit a
  single label for the entire input (job 3).

Same layer. Different thing taken out of it.


In [ ]:
# ================== TRY IT 1 ==================
# Change the batch to 7 sequences of 20 steps. Which of the two shapes
# changes, and which does not?
# ==============================================


## Part 2. Job 1: generate a headline


A language model scores every possible next word at every step. To generate, take
the scores at the last step, sample a word, feed it back in, and repeat.

This one read the sarcastic half of the corpus and nothing else.


In [ ]:
class Generator(nn.Module):
    """Job 1. One output per step, scoring every possible next word."""

    def __init__(self, n_words, hidden=160):
        super().__init__()
        self.embedding = nn.Embedding(n_words, hidden)
        self.lstm = nn.LSTM(hidden, hidden, batch_first=True)
        self.head = nn.Linear(hidden, n_words)
        self.head.weight = self.embedding.weight       # one matrix, used twice

    def forward(self, ids, state=None):
        outputs, state = self.lstm(self.embedding(ids), state)
        return self.head(outputs), state          # <-- one score per STEP


saved = torch.load("data/generator.pt", weights_only=False)
GEN_WORDS, GEN_INDEX = saved["words"], saved["index"]
# Placeholder rows, plus the words a published satirical newspaper uses and a
# lecture theatre does not. Masked out below rather than removed from the model.
BLOCKED = saved["banned"]

generator = Generator(len(GEN_WORDS))
generator.load_state_dict(saved["state"])
generator.eval()

print("it knows", len(GEN_WORDS), "words")

In [ ]:
def invent_headline(start="area man", temperature=0.8, limit=14):
    """Sample one word at a time, feeding each one back in."""
    ids = torch.tensor([[GEN_WORDS.get(w, 1) for w in start.split()]])
    state = None
    words = start.split()
    with torch.no_grad():
        for step in range(limit):
            scores, state = generator(ids, state)
            row = scores[0, -1].clone()
            # Never sample a blocked row, and do not let it stop after two
            # words. Real decoders all do some version of this.
            row[BLOCKED] = -1e9
            if len(words) < 6:
                row[GEN_WORDS["<end>"]] = -1e9
            probabilities = torch.softmax(row / temperature, dim=0)
            nxt = int(torch.multinomial(probabilities, 1))
            if GEN_INDEX[nxt] == "<end>":
                break
            words.append(GEN_INDEX[nxt])
            ids = torch.tensor([[nxt]])
    return " ".join(words)


torch.manual_seed(0)
for opening in ["area man", "study finds", "nation's", "report:", "man who"]:
    print(" ", invent_headline(opening))


It has learned the shape of the joke: the deadpan register, the "area man", the
sudden specificity. It has not learned to be funny, because it is a 3 MB model
that read 700 KB of text.

That is worth naming. Everything about the architecture here is the same as a
modern language model. The difference is scale, and it is the whole difference.


In [ ]:
# ================== TRY IT 2 ==================
# Generate with `temperature=0.3` and with `temperature=1.5`.
# What is the number doing?
# ==============================================


## Part 3. Job 2: label every token


Same LSTM, same per-step outputs. The only change is what the head predicts: a
part-of-speech tag instead of a character.


In [ ]:
class Tagger(nn.Module):
    """Job 2. One output per step, scoring every possible tag."""

    def __init__(self, n_words, n_tags, hidden=128):
        super().__init__()
        self.embedding = nn.Embedding(n_words, 64, padding_idx=0)
        self.lstm = nn.LSTM(64, hidden, batch_first=True)
        self.head = nn.Linear(hidden, n_tags)

    def forward(self, ids):
        outputs, _ = self.lstm(self.embedding(ids))
        return self.head(outputs)                 # <-- one score per STEP


saved = torch.load("data/tagger.pt", weights_only=False)
TAG_WORDS, TAGS = saved["words"], saved["tags"]

tagger = Tagger(len(TAG_WORDS), len(TAGS))
tagger.load_state_dict(saved["state"])
tagger.eval()


def tag(sentence):
    words = sentence.lower().split()
    ids = torch.tensor([[TAG_WORDS.get(w, 1) for w in words]])
    with torch.no_grad():
        predicted = tagger(ids).argmax(-1)[0]
    return [(w, TAGS[t]) for w, t in zip(words, predicted.tolist())]


for word, label in tag("the committee approved the budget on friday"):
    print(f"  {word:12s} {label}")

In [ ]:
# ================== TRY IT 3 ==================
# Tag a sentence where the same word is used two different ways, such as
# "they book a flight to read the book". Does it get both right?
# ==============================================


## Part 4. Job 3: one label for the whole sequence


Now throw the per-step outputs away and keep only the final hidden state, which
has read everything. One vector in, one label out.


In [ ]:
class Classifier(nn.Module):
    """Job 3. ONE output for the whole sequence, from the final hidden state."""

    def __init__(self, n_words, hidden=128):
        super().__init__()
        self.embedding = nn.Embedding(n_words, 64, padding_idx=0)
        self.lstm = nn.LSTM(64, hidden, batch_first=True)
        self.head = nn.Linear(hidden, 2)

    def forward(self, ids):
        outputs, (last_hidden, last_cell) = self.lstm(self.embedding(ids))
        return self.head(last_hidden[-1])         # <-- one score per SEQUENCE


saved = torch.load("data/classifier.pt", weights_only=False)
CLASSIFIER_WORDS = saved["words"]

classifier = Classifier(len(CLASSIFIER_WORDS))
classifier.load_state_dict(saved["state"])
classifier.eval()


def is_sarcastic(text, model=classifier, words=CLASSIFIER_WORDS, width=25):
    ids = np.zeros((1, width), dtype=np.int64)
    for j, word in enumerate(text.split()[:width]):
        ids[0, j] = words.get(word, 1)
    with torch.no_grad():
        scores = model(torch.from_numpy(ids))
    return float(torch.softmax(scores, 1)[0, 1])


for text in ["supreme court rules on landmark voting rights case",
             "here's what happened when i slept for an extra hour each night",
             "man who has never cooked confident he could run restaurant",
             "area man passionate defender of what he imagines constitution to be",
             "study finds link between exercise and heart health"]:
    print(f"  {is_sarcastic(text):.2f}   {text}")
print()
print("The last one is real news and it scores 0.99. Look at the first four")
print("words: the model has decided that headlines shaped like a study finding")
print("something are Onion headlines, because in this corpus they usually are.")


### The three jobs, side by side

```
Job 1  generate      return self.head(outputs), state      one score per STEP
Job 2  tag           return self.head(outputs)             one score per STEP
Job 3  classify      return self.head(last_hidden[-1])     one score per SEQUENCE
```

Three architectures from the slides, and the difference between them is which
return value you reach for. Everything else is the same layer.



---

## Your turn: train a sarcasm detector

The model below is yours to design. It works, and it is barely better than
guessing. **Teams of three, and the scoreboard is on the board.**

<img src="images/activity-sarcasm.png" width="640">


In [ ]:
# GIVEN. Data and the training harness. You will not need to edit this cell.
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

MAX_WORDS = 25           # how many words of the headline are read
VOCAB_SIZE = 2000        # how many distinct words get their own row
EPOCHS = 1               # passes over the training set
LEARNING_RATE = 0.002    # how big each step downhill is

texts = headlines["headline"].astype(str).tolist()
targets = headlines["is_sarcastic"].tolist()
train_texts, test_texts, train_y, test_y = train_test_split(
    texts, targets, test_size=0.2, random_state=0, stratify=targets)


def make_vocabulary(rows, size):
    from collections import Counter
    counts = Counter(word for row in rows for word in row.split())
    vocabulary = {"<pad>": 0, "<unk>": 1}
    for word, _ in counts.most_common(size):
        vocabulary[word] = len(vocabulary)
    return vocabulary


def encode(rows, vocabulary):
    ids = np.zeros((len(rows), MAX_WORDS), dtype=np.int64)
    for i, row in enumerate(rows):
        for j, word in enumerate(row.split()[:MAX_WORDS]):
            ids[i, j] = vocabulary.get(word, 1)
    return torch.from_numpy(ids)


def train_and_score(build_model, seed=0):
    """Train one architecture on the headlines and return its test accuracy."""
    torch.manual_seed(seed)
    vocabulary = make_vocabulary(train_texts, VOCAB_SIZE)
    x, y = encode(train_texts, vocabulary), torch.tensor(train_y)
    model = build_model(vocabulary)
    optimiser = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    loss_function = nn.CrossEntropyLoss()
    for epoch in range(EPOCHS):
        order = torch.randperm(len(y))
        for start in range(0, len(y), 64):
            batch = order[start:start + 64]
            optimiser.zero_grad()
            loss_function(model(x[batch]), y[batch]).backward()
            optimiser.step()
    with torch.no_grad():
        predicted = model(encode(test_texts, vocabulary)).argmax(1).numpy()

    # Keep the trained model around so you can look at its mistakes later.
    global LAST_MODEL, LAST_VOCABULARY
    LAST_MODEL, LAST_VOCABULARY = model, vocabulary
    return round(accuracy_score(test_y, predicted), 3)


print("train_and_score(MyLSTM) is ready.")
print("always guessing the majority class scores",
      round(max(np.mean(test_y), 1 - np.mean(test_y)), 3))


### Ideas

All of these are changes to the **code**, not to a setting.

1. **Make it wider.** Eight numbers is not much of a memory.
2. **Read it both ways.** `nn.LSTM(..., bidirectional=True)` runs a second LSTM
   backwards over the sequence. Then `last_hidden` has two rows instead of one,
   so join them: `torch.cat([last_hidden[0], last_hidden[1]], dim=1)`, and your
   head takes twice as many inputs.
3. **Stack two of them.** `nn.LSTM(..., num_layers=2)`.
4. **Use the per-step outputs instead**, averaged, rather than the final state.
5. **Add `nn.Dropout`** between the LSTM and the head.
6. **Change the four settings** at the top of the given cell.


In [ ]:
# ================== YOUR TURN 1 ==================
# Rewrite `MyLSTM` into something that actually works, then run the cell.
#
# The only fixed points are the argument to `forward` and the two output
# scores.
#
# Expected: the model as written scores 0.675. Always guessing the majority
#           class gets 0.562, counting words with logistic regression gets 0.840,
#           and the trained model in Part 4 gets 0.843. One run takes about a
#           second, so try a lot of things.
# =================================================
class MyLSTM(nn.Module):
    """Eight numbers per word, an eight-wide memory, one linear head."""

    def __init__(self, vocabulary):
        super().__init__()
        self.embedding = nn.Embedding(len(vocabulary), 8, padding_idx=0)
        self.lstm = nn.LSTM(8, 8, batch_first=True)
        self.head = nn.Linear(8, 2)

    def forward(self, ids):
        outputs, (last_hidden, last_cell) = self.lstm(self.embedding(ids))
        return self.head(last_hidden[-1])


print("test accuracy:", train_and_score(MyLSTM))

In [ ]:
# ================== YOUR TURN 2 ==================
# Look at what it gets wrong.
#
# Print ten headlines your model misclassified. Are they unfair, or would
# you have got them wrong too?
#
# Expected: mostly genuinely ambiguous headlines: deadpan real news that reads
#           like satire, and Onion headlines that are only odd if you know the
#           subject. That is the ceiling on this task, not a bug in your model.
# =================================================
# LAST_MODEL is whatever train_and_score trained most recently.
with torch.no_grad():
    predicted = LAST_MODEL(encode(test_texts, LAST_VOCABULARY)).argmax(1).numpy()

wrong = [(t, y) for t, y, p in zip(test_texts, test_y, predicted) if y != p]
print(len(wrong), "wrong out of", len(test_texts))
print()
for text, truth in wrong[:10]:
    print(f"  really {'sarcastic' if truth else 'not sarcastic'}:  {text}")

## Answers

Try each task before reading.

In [ ]:
# TRY IT 1
#   outputs becomes (7, 20, 32) and last_hidden stays (1, 7, 32).
#   The time axis only exists in outputs. last_hidden is one vector per
#   sequence no matter how long the sequence was, which is exactly why it is
#   the thing you hand to a classifier.

# TRY IT 2
#   Temperature divides the scores before the softmax. Low (0.3) sharpens the
#   distribution, so the model keeps picking its single most likely next word
#   and falls into stock phrases it saw often. High (1.5) flattens it, rare
#   words get picked, and the headline stops making sense. 0.8 is a usable
#   middle. This is the same dial you will meet again as a decoding parameter
#   on a real language model.

# TRY IT 3
#   It usually gets both, because the tag is predicted from the hidden state,
#   which has read the words before it. "they book a flight" tags book as a
#   verb; "read the book" tags it as a noun. A tagger that looked at one word
#   at a time could not do this, and that is the whole argument for reading in
#   order.

# YOUR TURN 1
#   Measured on the test split, one change at a time from the starting model:
#
#     always guess the common label                   0.562
#     the model as given (8 wide, 1 epoch)            0.675
#     + EPOCHS 4                                      0.779
#     + bidirectional                                 0.742
#     + mean of the per-step outputs                  0.737
#     + VOCAB_SIZE 10000                              0.668
#     + wider: embedding 64, hidden 128               0.617   <- WORSE
#
#   Stacking, and this is where it gets interesting:
#
#     wide + EPOCHS 4                                 0.562   <- collapsed
#       + VOCAB_SIZE 10000                            0.844   <- best
#       + bidirectional                               0.841
#       + dropout 0.3                                 0.837
#       + two layers                                  0.837
#
#     counting words + logistic regression            0.840
#     the trained model in Part 4                     0.843
#
#   Notice that this is NOT a tidy ladder. Widening the model on its own made it
#   worse, and widening it with four epochs collapsed it to guessing, and then
#   one more change took the same model to the best score on the board. Small
#   LSTMs are unstable: the same architecture can land in a good or a useless
#   place depending on where it started. If a change makes things worse for no
#   reason you can see, run it again with a different seed before you believe it.
#
#   The best model found here:
#
#     class MyLSTM(nn.Module):
#         def __init__(self, vocabulary):
#             super().__init__()
#             self.embedding = nn.Embedding(len(vocabulary), 64, padding_idx=0)
#             self.lstm = nn.LSTM(64, 128, batch_first=True)
#             self.head = nn.Linear(128, 2)
#
#         def forward(self, ids):
#             outputs, (last_hidden, last_cell) = self.lstm(self.embedding(ids))
#             return self.head(last_hidden[-1])
#
#     with VOCAB_SIZE = 10000 and EPOCHS = 4.
#
# YOUR TURN 2
#   The mistakes are mostly real ambiguity. Deadpan political reporting reads
#   exactly like satire, and plenty of Onion headlines are only funny if you
#   recognise the reference. Read three out loud and ask the room to vote before
#   revealing the label; the room does not get 100% either.

# The three things worth carrying out of today:
#   1. An LSTM hands you one vector per step and one vector for the sequence.
#      Which one you use IS the architecture.
#   2. Reading in order buys you things a bag of words cannot have: the tagger
#      gets "book" right twice in the same sentence.
#   3. Bidirectional is nearly free and usually helps, because the end of a
#      sentence often decides what the beginning meant.